# Explainable Pneumonia Screening with Anatomy-Grounded Reports
**Base:** TorchXRayVision (cloned) · **Our addition:** light fine-tuning on Kermany + a coarse-to-fine, measurement-grounded report generator.

Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

## 1. Setup: clone the base repo and install

In [ ]:
!git clone -q --depth 1 https://github.com/mlmed/torchxrayvision.git
!pip -q install torchxrayvision reportlab streamlit kagglehub
# Upload project.zip with the Files panel (left sidebar), then:
!unzip -o -q project.zip -d project
%cd project
!ls

## 2. Download the Kermany chest X-ray dataset (5,856 pediatric images)

In [ ]:
import kagglehub, glob, os
root = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
DATA = [os.path.dirname(os.path.dirname(p)) for p in glob.glob(f"{root}/**/test/NORMAL", recursive=True)
        if "__MACOSX" not in p][0]
print("DATA =", DATA)
for s in ["train", "test"]:
    print(s, {c: len(os.listdir(f"{DATA}/{s}/{c}")) for c in ["NORMAL", "PNEUMONIA"]})

## 3. Light fine-tuning on Kermany (≈5–8 min)
Backbone frozen (adult RSNA DenseNet-121), new head trained on pediatric features, threshold tuned for **recall ≥ 0.95**.

In [ ]:
!python finetune.py --data "$DATA" --out weights --epochs 30

## 4. Evaluate on the 624-image independent test set (numbers for the slides)

In [ ]:
!python evaluate.py --data "$DATA" --head weights/kermany_head.pt
from IPython.display import Image as IPImage, display
display(IPImage("results/confusion.png"))

## 5. Generate example reports for the presentation

In [ ]:
import random
from report_engine import Engine, generate_report
from report_pdf import make_pdf, make_panel
eng = Engine(head_path="weights/kermany_head.pt")
os.makedirs("example_reports", exist_ok=True)
random.seed(7)
picks = (random.sample(glob.glob(f"{DATA}/test/PNEUMONIA/*"), 4)
         + random.sample(glob.glob(f"{DATA}/test/NORMAL/*"), 3))
for p in picks:
    m, vis = eng.analyse(p)
    rep = generate_report(m)
    name = os.path.basename(os.path.dirname(p)) + "_" + os.path.splitext(os.path.basename(p))[0]
    make_pdf(rep, m, vis, f"example_reports/{name}.pdf", patient_id=name, model_name=eng.main_name)
    make_panel(vis, m, f"example_reports/{name}.png")
    print(f"\n=== {name}   (ground truth: {name.split('_')[0]})")
    for k, v in rep["sections"].items():
        print(f"  {k}: {v}")

In [ ]:
for p in sorted(glob.glob("example_reports/*.png"))[:3]:
    display(IPImage(p))

## 6. Download everything (weights for the demo app + results + example PDFs)

In [ ]:
!zip -q -r outputs.zip weights results example_reports
from google.colab import files
files.download("outputs.zip")

## 7. Run the demo app
**Easiest:** on your laptop, put `weights/` next to `app.py` and run `streamlit run app.py` (CPU is fine).

**Or from Colab** with a public link (open the loca.lt URL, paste the password it prints):

In [ ]:
!npm -q install -g localtunnel
!streamlit run app.py --server.headless true &>/content/st.log &
!sleep 8 && curl -s https://loca.lt/mytunnelpassword && echo "   <- tunnel password"
!npx localtunnel --port 8501